# 📊 E-Commerce Big Data — Monitoring & Observability Dashboard

**Step 8 of Big Data Architecture: Production Monitoring, Quality Auditing & Alerting**

Dashboard giám sát toàn diện hoạt động của hệ thống Big Data E-commerce, bao gồm:
- **Data Ingestion Health**: Lịch sử thực thi, throughput, thời gian xử lý và tỷ lệ lỗi.
- **Data Quality Audits**: Điểm số hoàn thiện (Completeness), phiên hợp lệ (Session Validity) và tỷ lệ ngoại lai (Outliers).
- **Model Registry & Drift**: Trạng thái các mô hình (K-Means, Random Forest, Recommender) và metrics đánh giá.
- **Host Resource Telemetry**: Mức chiếm dụng RAM và Dung lượng ổ đĩa.
- **Automated Alerts**: Cảnh báo tức thì khi có sự cố hoặc suy giảm chất lượng dữ liệu.

## 1. Setup & Khởi tạo Môi trường Giám Sát

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DB_PATH, MODELS_DIR
from src.storage import get_connection, get_database_stats
from src.monitoring import (
    get_pipeline_history,
    get_quality_reports,
    get_model_registry_summary,
    get_system_health,
    evaluate_system_alerts
)

print("✅ Monitoring suite successfully initialized!")

## 2. Executive KPI & Infrastructure Health Summary

In [ ]:
health = get_system_health()
db = health['database']
sys_res = health['system_resources']

print("="*65)
print("           BIG DATA INFRASTRUCTURE HEALTH AUDIT")
print("="*65)
print(f"Database Status        : {'ONLINE' if db['exists'] else 'OFFLINE'}")
print(f"Storage Location       : {DB_PATH}")
print(f"Total Database Size    : {db['size_mb']:.2f} MB")
print(f"Analytical Events Rows : {db['total_events']:,}")
print(f"System Memory (RAM)    : {sys_res['ram_usage_percent']}% used ({sys_res['ram_available_gb']} GB available)")
print(f"Disk Free Space        : {sys_res['disk_free_gb']} GB ({sys_res['disk_usage_percent']}% utilized)")
print("="*65)

## 3. Database Table Distribution & Volume

In [ ]:
if db['tables']:
    tables_df = pd.DataFrame(list(db['tables'].items()), columns=['Table Name', 'Row Count'])
    tables_df = tables_df.sort_values('Row Count', ascending=False).reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=tables_df, x='Row Count', y='Table Name', palette='Blues_r', ax=ax)
    ax.set_xscale('log')
    ax.set_title('Database Table Record Volumes (Log Scale)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Record Count (Logarithmic)')
    for i, count in enumerate(tables_df['Row Count']):
        ax.text(count * 1.1, i, f"{count:,}", va='center', fontsize=9, fontweight='semibold')
    plt.tight_layout()
    plt.show()
    
    display(tables_df)
else:
    print("No tables found in database. Please run pipeline first.")

## 4. Pipeline Execution History & Latency Tracking (`pipeline_metadata`)

In [ ]:
pipeline_df = get_pipeline_history(limit=15)

if not pipeline_df.empty:
    display(pipeline_df)
    
    # Visualize Execution Duration per Source File
    fig, ax = plt.subplots(figsize=(11, 4))
    sns.barplot(data=pipeline_df, x='source_file', y='execution_seconds', hue='status', palette={'SUCCESS': '#2ecc71', 'QUALITY_WARNING': '#f39c12', 'FAILED': '#e74c3c'}, ax=ax)
    ax.set_title('ETL Processing Latency per Ingested Batch (Seconds)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Execution Time (s)')
    ax.set_xlabel('Source File')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Pipeline metadata is empty. Run ETL pipeline to populate execution metrics.")

## 5. Automated Data Quality Audits (`data_quality_reports`)

In [ ]:
quality_df = get_quality_reports(limit=15)

if not quality_df.empty:
    display(quality_df)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # Completeness Score Trend
    ax1.plot(quality_df['source_file'], quality_df['completeness_score'] * 100, marker='o', color='#3498db', linewidth=2)
    ax1.axhline(95.0, color='r', linestyle='--', label='Min Quality Gate (95%)')
    ax1.set_title('Data Completeness Score Trend (%)', fontweight='bold')
    ax1.set_ylabel('Completeness %')
    ax1.set_ylim(85, 102)
    ax1.tick_params(axis='x', rotation=25)
    ax1.legend()
    
    # Session Validity Rate
    ax2.plot(quality_df['source_file'], quality_df['valid_sessions_rate'] * 100, marker='s', color='#2ecc71', linewidth=2)
    ax2.set_title('Valid Session Identification Rate (%)', fontweight='bold')
    ax2.set_ylabel('Valid Session %')
    ax2.set_ylim(85, 102)
    ax2.tick_params(axis='x', rotation=25)
    
    plt.tight_layout()
    plt.show()
else:
    print("Quality report table empty.")

## 6. Model Registry & Performance Metrics

In [ ]:
registry = get_model_registry_summary()

if registry.get('models'):
    models_table = []
    for m in registry['models']:
        models_table.append({
            'Model Name': m.get('model_name'),
            'Version': m.get('version'),
            'Algorithm': m.get('algorithm'),
            'Key Metrics': json.dumps(m.get('metrics')),
            'Artifact Path': m.get('artifact_path')
        })
    display(pd.DataFrame(models_table))
else:
    print("Model registry has no registered models.")

## 7. Automated Alert System

In [ ]:
alerts = evaluate_system_alerts()

if not alerts:
    print("✅ ALL SYSTEMS HEALTHY: No operational warnings or critical alerts detected.")
else:
    print(f"⚠️  {len(alerts)} ACTIVE ALERTS DETECTED:")
    for a in alerts:
        icon = '🔴' if a['severity'] == 'CRITICAL' else ('🟠' if a['severity'] == 'HIGH' else '🟡')
        print(f"{icon} [{a['severity']}] Component: {a['component']}")
        print(f"   Message: {a['message']}\n")